## Patient Journey - PX
#### Data source used
* PHCDW.PHCDW_PUB_CAR.PUB_CAR_IQV_LAAD_PX_FCT (PHCDW.PHCDW_DSAA.IQV_LAAD_PX_VW was used before 27th Oct, 2022)
* PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PROCEDURE

#### Steps:
1. Query px claims data
2. Get first dialysis date for each patient
3. Save table to databricks

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

### Load Libraries

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
msal 1.36.0 requires cryptography<49,>=2.5, but you have cryptography 50.0.2 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run "./00_cohort_config"

### Get Data from Scope

[REDACTED]

[REDACTED]

[REDACTED]

[REDACTED]



# Cohort Configuration

Shared config notebook for the 6 patient journey validation notebooks. 
Defines the offset, data boundaries, cohort labels (C1–C22), and the cohort assignment function.

**Usage:** Each patient journey notebook should `%run` this notebook early on.

### Define Snowflake Connection Function
Choose appropiate below items based on the Brand
- Role
- Warehouse
- Schema
- Database

OFFSET_DAY      = 7
DATA_START_DATE = 2023-01-01
COHORT_ORIGIN   = 2025-01-01
COHORT_LAST     = 2026-10-01


In [0]:


query = """
      SELECT DISTINCT PATIENT_ID,
                      min(SERVICE_DATE) as FIRST_DIALYSIS_DATE,    
                      1 as DIALYSIS_FLAG
                      --PX.PROCEDURE_CODE,
                      --PROCEDURE_DESCRIPTION
                      
      FROM (select PATIENT_ID, SERVICE_DATE, PROCEDURE_CODE from PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FACT_PX)  PX
      JOIN 
            (   SELECT DISTINCT  PROCEDURE_CODE,PROCEDURE_DESCRIPTION
                FROM PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PROCEDURE 
                WHERE UPPER(PROCEDURE_DESCRIPTION) LIKE '%DIALYSIS%'
            ) PXCD
      ON PX.PROCEDURE_CODE = PXCD.PROCEDURE_CODE
      group by PATIENT_ID, DIALYSIS_FLAG
   """  
df_px = Get_Data_Snowflakes(query)
df_px.display()

PATIENT_ID,FIRST_DIALYSIS_DATE,DIALYSIS_FLAG
1408831340,2023-03-29,1
2761988966,2022-04-01,1
330182925,2021-04-01,1
1458146137,2021-01-02,1
1699083172,2024-01-01,1
1252380223,2022-09-01,1
1571688682,2021-01-02,1
1708093833,2021-01-03,1
158175206,2021-01-20,1
1731023916,2023-07-16,1


cohort_label,cohort_ref_month,cohort_start,cohort_end
C1,2025-01-01,2025-01-07,2025-02-07
C2,2025-02-01,2025-02-07,2025-03-07
C3,2025-03-01,2025-03-07,2025-04-07
C4,2025-04-01,2025-04-07,2025-05-07
C5,2025-05-01,2025-05-07,2025-06-07
C6,2025-06-01,2025-06-07,2025-07-07
C7,2025-07-01,2025-07-07,2025-08-07
C8,2025-08-01,2025-08-07,2025-09-07
C9,2025-09-01,2025-09-07,2025-10-07
C10,2025-10-01,2025-10-07,2025-11-07



22 cohorts defined (C1 – C22)


Testing with current OFFSET_DAY = 7
-------------------------------------------------------
  2025-01-07 → 2025-01-01  (expected 2025-01-01)  [PASS]
  2025-01-06 → 2024-12-01  (expected 2024-12-01)  [PASS]
  2025-02-06 → 2025-01-01  (expected 2025-01-01)  [PASS]
  2025-02-07 → 2025-02-01  (expected 2025-02-01)  [PASS]
  2025-10-04 → 2025-09-01  (expected 2025-09-01)  [PASS]

Config notebook ready. All 6 patient journey notebooks can %run this.


In [0]:
# VALIDATION: write to a separate path so we don't overwrite production data
df_px.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_px_validation")